In [ ]:
!pip install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import imblearn
import matplotlib.pyplot as plt
plt.rcParams['text.usetex'] = True

In [ ]:
data_random = pd.read_csv('../data/random.dat', names=['C1', 'C3', 'bq', 'likelihood'])
data_plot = pd.read_csv('../data/plotdata.csv')

In [ ]:
drandom = data_random.loc[data_random['likelihood']>-50]
dplot = data_plot.loc[data_plot['likelihood']>-50]

In [ ]:
data = pd.concat([drandom, dplot], ignore_index=True)

We will oversample the data with medium likelihood, because they are too few points in this range

In [ ]:
groups = []
for i in data.index:
    if data['likelihood'][i] < 16:
        groups.append(0)
    elif data['likelihood'][i] < 18:
        groups.append(1)
    #elif drandom['likelihood'][i] < 20:
    #    groups.append(2)
    else:
        groups.append(3)

groups = pd.Series(groups, index=data.index)

In [ ]:
sm = imblearn.over_sampling.SMOTE()
dataset, ysmote = sm.fit_resample(data, groups)

In [ ]:
import pandas as pd
import numpy as np
import imblearn
import matplotlib.pyplot as plt
plt.rcParams['text.usetex'] = True

In [ ]:
import sklearn.model_selection


training, validation = sklearn.model_selection.train_test_split(dataset)

In [ ]:
plt.scatter(dataset['C1'], dataset['C3'], c=dataset['likelihood'], cmap='rainbow', s=0.3)

In [ ]:
plt.scatter(dataset['C1'], dataset['bq'], c=dataset['likelihood'], cmap='rainbow', s=0.3)

In [ ]:
plt.scatter(dataset['C3'], dataset['bq'], c=dataset['likelihood'], cmap='rainbow', s=0.3)

In [ ]:
trainX = training[['C1', 'C3', 'bq']]

In [ ]:
trainY = training['likelihood']

In [ ]:
valX = validation[['C1', 'C3', 'bq']]
valY = validation['likelihood']

In [ ]:
import xgboost

In [ ]:
import xgboost.callback


earlystop = xgboost.callback.EarlyStopping(5, data_name='validation_0', save_best=True)

xr = xgboost.XGBRegressor(n_estimators=3000, callbacks=[earlystop], learning_rate=0.03)
xr.fit(trainX, trainY, eval_set=[(valX, valY)])

In [ ]:
xr.save_model('xgboost_scIII.json')

In [ ]:
def predict_point(C1, C3, bq):
    if C1 > 0:
        fC1 = np.exp(C1*50)-1
    elif C1 < -0.3:
        fC1 = np.exp((-C1-0.3)*50)-1
    else:
        fC1 = 0.0
    if C3 > 0:
        fC3 = np.exp(C3*50)-1
    elif C3 < -0.3:
        fC3 = np.exp((-C3-0.3)*50)-1
    else:
        fC3 = 0.0
    if bq < 0:
        fbq = np.exp(-bq*50)-1
    elif bq > 3.2:
        fbq = np.exp((bq-3.2)*50)-1
    else:
        fbq = 0.0
    return xr.predict(pd.DataFrame([{'C1': C1, 'C3': C3, 'bq': bq},]))[0]-fC1-fC3-fbq

In [ ]:
xgboost.plot_importance(xr, importance_type="weight")

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(xr, feature_names=['C1', 'C3', 'bq'])


In [ ]:
shapval = explainer(valX)

In [ ]:
shap.plots.scatter(shapval[:, "C1"],dot_size=4)


In [ ]:
shap.plots.scatter(shapval[:, "C3"],dot_size=4)

In [ ]:
shap.plots.scatter(shapval[:, "bq"], dot_size=4)

In [ ]:
shap.plots.beeswarm(shapval)


In [ ]:
# --- 0) Imports ---
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

from imblearn.over_sampling import SMOTE

import xgboost
from xgboost import XGBRegressor, callback

import shap
import matplotlib.pyplot as plt

# --- 1) Carga y filtro ---
# Ajusta las rutas si no están en la misma carpeta
data_random = pd.read_csv('srandom.dat', names=['C1','C3','bq','likelihood'])
data_plot   = pd.read_csv('../data/plotdata.csv')  # si no tiene cabeceras y tiene 4 cols, añade names=...

drandom = data_random.loc[data_random['likelihood'] > -50].copy()
dplot   = data_plot.loc[data_plot['likelihood'] > -50].copy()
data    = pd.concat([drandom, dplot], ignore_index=True)

# --- 2) Bins para equilibrar (solo para SMOTE/estratificación, no para el modelo) ---
def bin_group(v):
    if v < 16:   return 0
    elif v < 18: return 1
    else:        return 3
data['group'] = data['likelihood'].apply(bin_group)

# --- 3) Split primero (evita leakage), estratificado por 'group' ---
X = data[['C1','C3','bq']]
y = data['likelihood']
g = data['group']

Xtr, Xva, ytr, yva, gtr, gva = train_test_split(
    X, y, g,
    test_size=0.25,
    random_state=42,
    stratify=g
)

# --- 4) SMOTE SOLO en TRAIN y SOLO en FEATURES ---
sm = SMOTE(random_state=42)
Xtr_res, gtr_res = sm.fit_resample(Xtr, gtr)

# --- 5) Asignar y a los sintéticos con KNN (entrenado solo en train original) ---
knn = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn.fit(Xtr, ytr)
ytr_res = knn.predict(Xtr_res)

# --- 6) XGBoost con early stopping ---
earlystop = callback.EarlyStopping(rounds=50, data_name='validation_0', save_best=True)
model = XGBRegressor(
    n_estimators=1500,       # puedes subir a 3000 si tienes tiempo
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse'
)
model.fit(
    Xtr_res, ytr_res,
    eval_set=[(Xva, yva)],
    callbacks=[earlystop],
    verbose=False
)

# --- 7) Métrica de validación ---
yhat_val = model.predict(Xva)
rmse = mean_squared_error(yva, yhat_val, squared=False)
print(f"RMSE (validation): {rmse:.5f}")

# --- 8) Guardar modelo ---
model.save_model('xgboost_scIII.json')
print("Modelo guardado en: xgboost_scIII.json")

# --- 9) Feature importance (XGBoost) ---
fig = plt.figure()
xgboost.plot_importance(model, importance_type="weight")
plt.title("XGBoost Feature Importance (weight)")
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=200)
plt.close(fig)
print("Feature importance -> feature_importance.png")

# --- 10) SHAP: TreeExplainer y plots ---
# Para acelerar, usa un subconjunto de valid si es grande
subsample_size = min(1000, len(Xva))
Xva_sub = Xva
explainer = shap.TreeExplainer(model, feature_names=['C1','C3','bq'])
shap_val = explainer(Xva_sub)

# (a) Scatter por feature
for feat in ['C1','C3','bq']:
    fig = plt.figure()
    shap.plots.scatter(shap_val[:, feat], show=False)
    plt.title(f"SHAP Scatter — {feat}")
    plt.tight_layout()
    plt.savefig(f'shap_scatter_{feat}.png', dpi=200)
    plt.close(fig)
    print(f"SHAP scatter {feat} -> shap_scatter_{feat}.png")

# (b) Beeswarm global
fig = plt.figure()
shap.plots.beeswarm(shap_val, show=False, max_display=20)
plt.title("SHAP Beeswarm — validation")
plt.tight_layout()
plt.savefig('shap_beeswarm.pdf', dpi=200)
plt.close(fig)
print("SHAP beeswarm -> shap_beeswarm.pdf")
import numpy as np
import pandas as pd

# shap_vals = explainer(Xva)   # mismo Xva que usas para validar
shap_importance = np.abs(shap_val.values).mean(axis=0)  # impacto promedio |SHAP|
shap_imp_series = pd.Series(shap_importance, index=shap_val.feature_names).sort_values(ascending=False)
print(shap_imp_series)


In [ ]:
def plot_shap_scatter_nohist(shap_exp, feature_name, out_path, s=6, alpha=0.6):
    # shap_exp[:, feature_name] devuelve una Explanation por-feature
    per_feat = shap_exp[:, feature_name]
    x_data   = per_feat.data    # valores de la feature
    y_shap   = per_feat.values  # valores SHAP (impacto)
    plt.figure()
    plt.scatter(x_data, y_shap, s=s, alpha=alpha)
    plt.xlabel(feature_name,fontsize=16)
    plt.ylabel("SHAP value",fontsize=16)
    plt.title(f"SHAP scatter — {feature_name}")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# Uso (asumiendo que ya tienes 'explainer' y 'Xva_sub' creados):
# shap_val = explainer(Xva_sub)

plot_shap_scatter_nohist(shap_val, "C1", "shap_scatter_C1.pdf")
plot_shap_scatter_nohist(shap_val, "C3", "shap_scatter_C3.pdf")
plot_shap_scatter_nohist(shap_val, "bq", "shap_scatter_bq.pdf")

In [ ]:
def plot_shap_scatter_nohist_byindex(shap_exp, feat_idx, out_path,
                                     xlabel=None, s=6, alpha=0.6):
    per_feat = shap_exp[:, feat_idx]
    x_data   = per_feat.data
    y_shap   = per_feat.values
    plt.figure()
    plt.scatter(x_data, y_shap, s=s, alpha=alpha, edgecolors='none')
    plt.xlabel(xlabel if xlabel is not None else shap_exp.feature_names[feat_idx],fontsize=18)
    plt.ylabel("SHAP value",fontsize=18)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# Si el orden de tus features es [C1, C3, bq], bq = índice 2:
plot_shap_scatter_nohist_byindex(shap_val, 2, "SHAPbq.pdf", xlabel=r'$\beta^q$')
plot_shap_scatter_nohist_byindex(shap_val, 0, "SHAPC1.pdf", xlabel=r'$C_1$')
plot_shap_scatter_nohist_byindex(shap_val, 1, "SHAPC3.pdf", xlabel=r'$C_3$')

# asumiendo que ya tienes:
# explainer = shap.TreeExplainer(model)
# shap_vals = explainer(Xva)   # Xva con columnas ['C1','C3','bq']

pretty_names = [r'$C_1$', r'$C_3$', r'$\beta^q$']   # reemplaza 'bq' por β_q

shap_vals_pretty = shap.Explanation(
    values=shap_val.values,
    base_values=shap_val.base_values,
    data=shap_val.data,
    feature_names=pretty_names
)

import matplotlib.pyplot as plt
plt.figure()
shap.plots.beeswarm(shap_vals_pretty, show=False, max_display=20)
plt.tight_layout()
plt.savefig("shap_beeswarm.pdf", dpi=200)
plt.close()


In [ ]:
import numpy as np
import shap

def compute_shap_row(model, X, y=None, idx=0):
    """
    Devuelve: base_value (float), shap_vals (np.array en orden X.columns),
              pred (float), actual (float o None)
    """
    explainer = shap.TreeExplainer(model)
    x_row = X.iloc[[idx]]                   # mantiene DataFrame 1xF
    exp = explainer(x_row)                  # SHAP para 1 muestra
    base = float(np.ravel(exp.base_values)[0])
    shap_vals = np.ravel(exp.values)        # contribuciones por feature (con signo), orden = X.columns
    pred = float(model.predict(x_row)[0])
    actual = None if y is None else float(np.ravel(y)[idx])
    return base, shap_vals, pred, actual

# === USO ===
idx = 0  # cambia al que quieras
base, shap_vals, pred, actual = compute_shap_row(model, Xva, y=yva, idx=idx)

# Imprime los números listos para copiar
print("Base value:", f"{base:.3f}")
for name, v in zip(Xva.columns, shap_vals):
    print(f"SHAP({name}): {v:.3f}")
print("Final prediction:", f"{pred:.3f}")
print("Actual:", "--" if actual is None else f"{actual:.3f}")
